<a href="https://colab.research.google.com/github/Nancy-44/Soil_degradation/blob/main/Soil_degradation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install earthengine-api cdsapi geopandas rasterio xarray rioxarray pandas numpy scikit-learn matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.7 MB/s eta 0:00:00
  Attempting uninstall: xarray
    Found existing installation: xarray 2025.12.0
    Uninstalling xarray-2025.12.0:
      Successfully uninstalled xarray-2025.12.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os

BASE="/content/drive/MyDrive/soil_degradation_project"

os.makedirs(BASE+"/raw",exist_ok=True)
os.makedirs(BASE+"/processed",exist_ok=True)


In [6]:
import ee
ee.Authenticate()
ee.Initialize(project= 'ndvi-489709')


In [7]:
"Définir la zone d'étude"
roi = ee.Geometry.Point([4.07,43.93]).buffer(5000)


In [11]:
"Calcul NDVI et NDWI"
def compute_indices(year):

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(f"{year}-01-01", f"{year}-12-31")
        .filterBounds(roi)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
    )

    def add_indices(img):
        ndvi = img.normalizedDifference(['B8','B4']).rename('NDVI')
        ndwi = img.normalizedDifference(['B3','B8']).rename('NDWI')
        return img.addBands([ndvi, ndwi])

    collection = collection.map(add_indices)

    ndvi_mean = collection.select('NDVI').mean()
    ndwi_mean = collection.select('NDWI').mean()

    return ndvi_mean, ndwi_mean


In [12]:
"Extraire NDVI et NDWI par année"
import pandas as pd

for year in range(2016,2024):

    ndvi, ndwi = compute_indices(year)

    ndvi_val = ndvi.reduceRegion(
        reducer = ee.Reducer.mean(),
        geometry = roi,
        scale = 10,
        maxPixels = 1e9
    ).getInfo()

    ndwi_val = ndwi.reduceRegion(
        reducer = ee.Reducer.mean(),
        geometry = roi,
        scale = 10,
        maxPixels = 1e9
    ).getInfo()

    print(year, ndvi_val, ndwi_val)



2016 {'NDVI': 0.5858271067515727} {'NDWI': -0.5811205308936497}
2017 {'NDVI': 0.5647774746909249} {'NDWI': -0.5763955070737389}
2018 {'NDVI': 0.5582508119940068} {'NDWI': -0.5544973934887165}
2019 {'NDVI': 0.5546143351743955} {'NDWI': -0.5515026687186041}
2020 {'NDVI': 0.5697376740335757} {'NDWI': -0.5560639208112813}
2021 {'NDVI': 0.5796803833115105} {'NDWI': -0.5675674618809176}
2022 {'NDVI': 0.5538237022881068} {'NDWI': -0.5642021643986068}
2023 {'NDVI': 0.5696921374679627} {'NDWI': -0.5705695491784982}


In [25]:
"Initialisation Copernicus"
cds_config = """
url: https://cds.climate.copernicus.eu/api
key: 6a3d8389-1f82-417d-be74-93fd2c13367c
"""

# Écrire le fichier dans /root/.cdsapirc
with open("/root/.cdsapirc", "w") as f:
    f.write(cds_config)




In [27]:
"Humidité de sol - ERA5-Land - station proche Sommières est Villevieille"
" Extraire par tranche de 5 ans et par paramètre car fichier lourd - période 1993-2024"

import cdsapi
import os

# Chemin où les fichiers NetCDF seront sauvegardés
output_dir = "./soil_moisture_era5/"
os.makedirs(output_dir, exist_ok=True)

c = cdsapi.Client()

years = list(range(1993, 2000))
months = [f"{m:02d}" for m in range(1, 13)]

for year in years:
    print(f"=== Téléchargement ERA5-Land Soil Moisture {year} ===")
    c.retrieve(
        'reanalysis-era5-land-monthly-means',
        {
            'variable': 'volumetric_soil_water_layer_1',  # couche 0-7cm
            'year': str(year),
            'month': months,
            'time': '00:00',
            'format': 'netcdf'
        },
        os.path.join(output_dir, f"soil_moisture_{year}.nc")
    )




=== Téléchargement ERA5-Land Soil Moisture 1993 ===


2026-03-10 09:31:32,404 INFO Request ID is 044e733c-6413-483f-8511-569cf69f1bc2
INFO:ecmwf.datastores.legacy_client:Request ID is 044e733c-6413-483f-8511-569cf69f1bc2
2026-03-10 09:31:32,562 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-03-10 09:31:47,751 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


d29ac6252c2cb715c8700486aa1e7df6.zip:   0%|          | 0.00/87.4M [00:00<?, ?B/s]

=== Téléchargement ERA5-Land Soil Moisture 1994 ===


2026-03-10 09:31:54,166 INFO Request ID is 2e0cc5df-ab1c-4553-9639-576e62189b2d
INFO:ecmwf.datastores.legacy_client:Request ID is 2e0cc5df-ab1c-4553-9639-576e62189b2d
2026-03-10 09:31:54,328 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-03-10 09:32:03,037 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-03-10 09:32:08,254 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


...

2026-03-10 09:34:44,718 INFO Request ID is aaa1e2b1-c4ba-40b0-af82-dfa20a2f5dfc
INFO:ecmwf.datastores.legacy_client:Request ID is aaa1e2b1-c4ba-40b0-af82-dfa20a2f5dfc
2026-03-10 09:34:44,880 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-03-10 09:34:53,596 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-03-10 09:35:35,374 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


1a5fbda49f6ede991a1e94278df85655.zip:   0%|          | 0.00/87.2M [00:00<?, ?B/s]

=== Téléchargement ERA5-Land Soil Moisture 1999 ===


2026-03-10 09:35:44,877 INFO Request ID is aebd73fd-7c78-4805-92b4-6babfd4fc82f
INFO:ecmwf.datastores.legacy_client:Request ID is aebd73fd-7c78-4805-92b4-6babfd4fc82f
2026-03-10 09:35:45,020 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-03-10 09:35:59,022 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-03-10 09:36:18,329 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


96654303104e8faba09438691207a96d.zip:   0%|          | 0.00/87.2M [00:00<?, ?B/s]

In [37]:
"Conversion format NetCDF => CSV"
import zipfile
import os

zip_folder = "/content/soil_moisture_era5"
extract_folder = "/content/soil_moisture_era5_extracted"

os.makedirs(extract_folder, exist_ok=True)

zip_files = [f for f in os.listdir(zip_folder) if f.endswith(".nc")]

for zf in zip_files:
    zip_path = os.path.join(zip_folder, zf)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)

print(" Extraction terminée. Fichiers extraits dans :", extract_folder)


✅ Extraction terminée. Fichiers extraits dans : /content/soil_moisture_era5_extracted


In [ ]:
import glob
import xarray as xr
import pandas as pd
import os

extract_folder = "/content/soil_moisture_era5_extracted"
nc_files = glob.glob(os.path.join(extract_folder, "*.nc"))
dfs = []

for f in nc_files:
    print("Traitement de :", f)
    ds = xr.open_dataset(f)  # maintenant c’est un vrai NetCDF
    # Remplacer 'sm' par le nom exact de la variable d'humidité du sol
    df = ds[['swvl1']].to_dataframe().reset_index()
    dfs.append(df)

if dfs:
    soil_df = pd.concat(dfs, ignore_index=True)
    soil_df.to_csv("/content/soil_moisture_5ans.csv", index=False)
    print("CSV créé")
else:
    print(" Aucun fichier traité")


Traitement de : /content/soil_moisture_era5_extracted/data_stream-moda.nc
